In [18]:
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import DecimalType
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()


Q7, dédoublonnage

la fonction dedoublonner(df)

In [19]:
def dedoublonner(df):
    fenetre = Window.partitionBy("id_commande").orderBy("_num_ligne")
    return (
        df
        .withColumn("rn", F.row_number().over(fenetre))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

In [20]:
dedup = dedoublonner(spark.table("lake.bronze.commandes_raw"))
print("Lignes après dédoublonnage :", dedup.count())

Lignes après dédoublonnage : 10000


Q8, typage et normalisation

conversion des types sans faire échouer le traitement

In [21]:
colonnes_decimales = ["prix_unitaire", "remise", "cout_livraison", "cout_marketing"]

typed = (
    dedup
    .withColumn("date_commande", F.col("date_commande").try_cast("date"))
    .withColumn("quantite", F.col("quantite").try_cast("int"))
)
for c in colonnes_decimales:
    typed = typed.withColumn(c, F.col(c).try_cast(DecimalType(10, 2)))

typed.printSchema()

root
 |-- id_commande: string (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- id_client: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- nom_produit: string (nullable = true)
 |-- prix_unitaire: decimal(10,2) (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- remise: decimal(10,2) (nullable = true)
 |-- cout_livraison: decimal(10,2) (nullable = true)
 |-- methode_paiement: string (nullable = true)
 |-- canal_marketing: string (nullable = true)
 |-- cout_marketing: decimal(10,2) (nullable = true)
 |-- statut_commande: string (nullable = true)
 |-- _source_fichier: string (nullable = true)
 |-- _date_ingestion: timestamp (nullable = true)
 |-- _num_ligne: integer (nullable = true)



correction de ville et jointure avec ref_villes

In [22]:
donnees = [
    ("Abidjan",     "Côte d'Ivoire",                    "CI"),
    ("Brazzaville", "Congo",                            "CG"),
    ("Cotonou",     "Bénin",                            "BJ"),
    ("Dakar",       "Sénégal",                          "SN"),
    ("Douala",      "Cameroun",                         "CM"),
    ("Kinshasa",    "République démocratique du Congo", "CD"),
    ("Libreville",  "Gabon",                            "GA"),
    ("Lomé",        "Togo",                             "TG"),
]

(
    spark.createDataFrame(donnees, ["ville", "pays", "code_pays"])
    .writeTo("lake.silver.ref_villes")
    .using("iceberg")
    .createOrReplace()
)

spark.table("lake.silver.ref_villes").show()

+-----------+--------------------+---------+
|      ville|                pays|code_pays|
+-----------+--------------------+---------+
|    Abidjan|       Côte d'Ivoire|       CI|
|Brazzaville|               Congo|       CG|
|    Cotonou|               Bénin|       BJ|
|      Dakar|             Sénégal|       SN|
|     Douala|            Cameroun|       CM|
|   Kinshasa|République démocr...|       CD|
| Libreville|               Gabon|       GA|
|       Lomé|                Togo|       TG|
+-----------+--------------------+---------+



In [23]:
ref_villes = spark.table("lake.silver.ref_villes").select("ville", "pays", "code_pays")

avec_pays = (
    typed
    .withColumn(
        "ville",
        F.when(F.trim(F.col("ville")) == "Kinshassa", "Kinshasa").otherwise(F.trim(F.col("ville"))),
    )
    .join(ref_villes, on="ville", how="left")
)

print("Lignes :", avec_pays.count())   # doit rester 10000

Lignes : 10000


vérification des pays nuls

In [24]:
avec_pays.createOrReplaceTempView("v_commandes")

spark.sql("""
    SELECT COUNT(*) AS lignes_sans_pays
    FROM v_commandes
    WHERE pays IS NULL
""").show()

+----------------+
|lignes_sans_pays|
+----------------+
|               0|
+----------------+



In [25]:
spark.sql("SELECT DISTINCT ville FROM v_commandes WHERE pays IS NULL").show()

+-----+
|ville|
+-----+
+-----+



catégorie et statut

In [26]:
nettoye = (
    avec_pays
    .withColumn("categorie_source", F.col("categorie"))
    .withColumn("categorie", F.split(F.col("nom_produit"), "_").getItem(1))
    .withColumn(
        "statut_commande",
        F.initcap(F.lower(F.trim(F.col("statut_commande")))),
    )
)

nettoye.groupBy("categorie").count().show()
nettoye.groupBy("statut_commande").count().show()

+------------+-----+
|   categorie|count|
+------------+-----+
|Électronique| 2770|
|      Beauté| 2027|
|      Maison| 2052|
|        Mode| 3151|
+------------+-----+

+---------------+-----+
|statut_commande|count|
+---------------+-----+
|        Annulée|  193|
|      Retournée|  816|
|         Livrée| 8991|
+---------------+-----+



Q9, corrections, indicateurs et quarantaine

appliquer les règles

In [27]:
prix_sentinelle = F.col("prix_unitaire") == -50
prix_negatif    = (F.col("prix_unitaire") < 0) & ~prix_sentinelle
remise_negative = F.col("remise") < 0
quantite_nulle  = F.col("quantite") == 0

corrige9 = (
    nettoye
    # 1. indicateurs, calculés AVANT de modifier les valeurs
    .withColumn("flag_prix_corrige",    F.coalesce(prix_negatif, F.lit(False)))
    .withColumn("flag_remise_corrigee", F.coalesce(remise_negative, F.lit(False)))
    # 2. motifs de rejet : un tableau, une ligne peut en avoir plusieurs
    .withColumn(
        "motifs_rejet",
        F.array_compact(F.array(
            F.when(prix_sentinelle, F.lit("PRIX_SENTINELLE")),
            F.when(quantite_nulle,  F.lit("QUANTITE_NULLE")),
        )),
    )
    # 3. corrections
    .withColumn("prix_unitaire",
        F.when(prix_negatif, F.abs(F.col("prix_unitaire"))).otherwise(F.col("prix_unitaire")))
    .withColumn("remise",
        F.when(remise_negative, F.lit(0).cast(DecimalType(10, 2))).otherwise(F.col("remise")))
)

séparer valides et rejets

In [28]:
rejets    = corrige9.filter(F.size("motifs_rejet") > 0)
commandes = corrige9.filter(F.size("motifs_rejet") == 0).drop("motifs_rejet")

comptages, taux de rejet et conservation

In [29]:
n_dedup = dedup.count()
n_ok    = commandes.count()
n_rej   = rejets.count()

print("lignes dédoublonnées :", n_dedup)
print("commandes            :", n_ok)
print("commandes_rejets     :", n_rej)
print(f"taux de rejet        : {n_rej / n_dedup:.2%}")
print("conservation         :", n_dedup == n_ok + n_rej)

lignes dédoublonnées : 10000
commandes            : 8836
commandes_rejets     : 1164
taux de rejet        : 11.64%
conservation         : True


In [30]:
rejets.select(F.explode("motifs_rejet").alias("motif")).groupBy("motif").count().show()   # 600 et 600
print(corrige9.filter("flag_prix_corrige").count())      # 22
print(corrige9.filter("flag_remise_corrigee").count())   # 600

+---------------+-----+
|          motif|count|
+---------------+-----+
| QUANTITE_NULLE|  600|
|PRIX_SENTINELLE|  600|
+---------------+-----+

22
600


Q10, colonnes calculées et MERGE INTO

ajouter les quatre colonnes

In [31]:
commandes = (
    commandes
    .withColumn("montant_brut", F.round(F.col("prix_unitaire") * F.col("quantite"), 2))
    .withColumn("ca_net", F.round(F.col("montant_brut") * (1 - F.col("remise")), 2))
    .withColumn(
        "marge_contribution",
        F.col("ca_net") - F.col("cout_livraison") - F.col("cout_marketing"),
    )
    .withColumn("annee_mois", F.date_format(F.col("date_commande"), "yyyy-MM"))
)

commandes.select("id_commande", "montant_brut", "ca_net", "marge_contribution", "annee_mois").show(5)

+-----------+------------+------+------------------+----------+
|id_commande|montant_brut|ca_net|marge_contribution|annee_mois|
+-----------+------------+------+------------------+----------+
|  CMD100000|      149.16|138.72|            135.27|   2025-12|
|  CMD100001|       74.36| 66.92|             51.99|   2025-07|
|  CMD100004|      997.42|977.47|            971.76|   2025-11|
|  CMD100005|      213.51|192.16|            178.08|   2025-10|
|  CMD100006|       69.12| 69.12|             56.30|   2025-07|
+-----------+------------+------+------------------+----------+
only showing top 5 rows


MERGE INTO

Sur cette plateforme, un MERGE INTO dont la source est une vue temporaire échoue (INTERNAL_ERROR de planification Spark Connect). On écrit donc d'abord le résultat dans la table intermédiaire lake.silver.commandes_stg, puis on fusionne depuis cette table.

In [32]:
# 1. tables intermédiaires : le calcul est écrit une fois
commandes.writeTo("lake.silver.commandes_stg").using("iceberg").createOrReplace()
rejets.writeTo("lake.silver.commandes_rejets").using("iceberg").createOrReplace()

# 2. table cible : créée vide seulement si elle n'existe pas
if not spark.catalog.tableExists("lake.silver.commandes"):
    (
        spark.table("lake.silver.commandes_stg").limit(0)
        .writeTo("lake.silver.commandes").using("iceberg").create()
    )

# 3. MERGE avec la table intermédiaire comme source
spark.sql("""
    MERGE INTO lake.silver.commandes AS t
    USING lake.silver.commandes_stg AS s
    ON t.id_commande = s.id_commande
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[]

In [33]:
print("commandes        :", spark.table("lake.silver.commandes").count())          # 8836
print("commandes_rejets :", spark.table("lake.silver.commandes_rejets").count())   # 1164

commandes        : 8836
commandes_rejets : 1164
